# 04 — World Bank Bronze to Silver

## 1. Read USA GDP Data

Read and parse World Bank GDP data for the United States from Bronze JSON.

In [ ]:
import json

path  = "Files/bronze/world_bank/worldbank_USA_NY.GDP.MKTP.CD_raw.json"

raw_json = notebookutils.fs.head(path, 100000)

data = json.loads(raw_json)

print("Elements:", len(data))
print("Metadata:", data[0])

In [ ]:
print(
    data[1][0]["countryiso3code"], 
    data[1][0]["date"],
    data[1][0]["value"])

In [ ]:
gdp_records = data[1]
for record in gdp_records[:3]:
    print(record["countryiso3code"], record["date"], record["value"])

## 2. Transform USA GDP Data

Extract country, year, and GDP values from JSON records and create a Spark DataFrame.

In [ ]:
gdp_clean = []

for record in gdp_records:
    gdp_clean.append({
        "country": record["countryiso3code"],
        "year": int(record["date"]),
        "gdp_usd": float(record["value"])
    }
    )

print("Total records:", len(gdp_clean))
print("First record:", gdp_clean[0])

In [ ]:
df_gdp = spark.createDataFrame(gdp_clean)

df_gdp.printSchema()
df_gdp.show(5)

## 3. Read Germany GDP Data

Read and parse World Bank GDP data for Germany (DEU) from Bronze JSON.

In [ ]:
import json

path  = "Files/bronze/world_bank/worldbank_DEU_NY.GDP.MKTP.CD_raw.json"

raw_json = notebookutils.fs.head(path, 100000)

data_germany = json.loads(raw_json)

print("Elements:", len(data_germany))
print("Metadata:", data_germany[0])

In [ ]:
gdp_germany_records = data_germany[1]

print("Germany records:", len(gdp_germany_records))
print("First record:", gdp_germany_records[0])

## 4. Transform Germany GDP Data

Extract country, year, and GDP values and create a Spark DataFrame.

In [ ]:
gdp_germany_clean = []

for record in gdp_germany_records:
    gdp_germany_clean.append({
        "country": record["countryiso3code"],
        "year": int(record["date"]),
        "gdp_usd": float(record["value"])
    }

    )


print("Germany records:", len(gdp_germany_clean))
print("First record:", gdp_germany_clean[0])

In [ ]:
df_gdp_germany = spark.createDataFrame(gdp_germany_clean)

df_gdp_germany.printSchema()
df_gdp_germany.show(5)
print("Germany rows:", df_gdp_germany.count())



## 5. Combine USA and Germany GDP Data

Combine GDP records from both countries into a single Spark DataFrame.

In [ ]:
df_gdp_combined = df_gdp.unionByName(df_gdp_germany)


print("Total rows:", df_gdp_combined.count())

df_gdp_combined.groupBy("country").count().show()

## 6. Data Quality Checks

Check duplicate country-year records, missing values, and invalid GDP amounts.

In [ ]:
unique_records = df_gdp_combined.dropDuplicates(
    ["country", "year"]
).count()

In [ ]:
total_rows = df_gdp_combined.count()

duplicates = total_rows - unique_records


print("Unique records:", unique_records)
print("Duplicates:", duplicates)

In [ ]:
null_gdp = df_gdp_combined.filter(df_gdp_combined.gdp_usd.isNull()).count()

print("NULL GDP:", null_gdp)

In [ ]:
null_country = df_gdp_combined.filter(df_gdp_combined.country.isNull()).count()

print("NULL Country:", null_country)

In [ ]:
null_year = df_gdp_combined.filter(df_gdp_combined.year.isNull()).count()

print("NULL year:", null_year)

In [ ]:
invalid_gdp = df_gdp_combined.filter(df_gdp_combined.gdp_usd <= 0).count()

print("Invalid GDP:", invalid_gdp)

## 7. Write Silver Delta Table

Save the combined USA and Germany GDP data as a Delta table in the Lakehouse.

In [ ]:
df_gdp_combined.write \
.format("delta")\
.mode("overwrite")\
.saveAsTable("silver_worldbank_gdp")

## 8. Validate Silver Output

Verify the saved table, row count, column count, and records per country.

In [ ]:
df_check = spark.table("silver_worldbank_gdp")

print("Rows:", df_check.count())
print("Columns:", len(df_check.columns))

df_check.groupBy("country").count().show()